In [1]:
!pip uninstall -y pillow
!pip install pillow==10.3.0

!pip install -U transformers accelerate sentencepiece

Found existing installation: pillow 12.2.0
Uninstalling pillow-12.2.0:
  Successfully uninstalled pillow-12.2.0
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 72.5 MB/s eta 0:00:00a 0:00:01
Defaulting to user installation because normal site-packages is not writeable


In [2]:
from PIL import Image
print(Image.__version__)
print(hasattr(Image, "Resampling"))

10.3.0
True


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "klyang/MentaLLaMA-chat-7B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("Model loaded successfully")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:01<00:00, 186.51it/s]


Model loaded successfully


[transformers] Error during conversion: ReadTimeout('The read operation timed out')


In [4]:
prompt = "I have been feeling very anxious lately and I don't know what to do."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.6,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I have been feeling very anxious lately and I don't know what to do. It seems like there are so many things going on in my life that are causing me stress and anxiety. I feel like I am constantly on edge and worried about something. I have tried to find ways to cope with my anxiety, such as exercise and meditation, but it seems like nothing is working. I am at a loss for what to do and I am feeling overwhelmed. Can you help me?




In [5]:
import pandas as pd

df = pd.read_csv("sampled_100_rows.csv")

print(df.shape)
df.head()

(100, 3)


,instruction,input,output
0,You are a helpful mental health counselling as...,I've been feeling anxious and on edge for week...,I understand that you're feeling anxious and o...
1,You are a helpful mental health counselling as...,I've been feeling guilty for not being able to...,I understand that you're feeling guilty for no...
2,You are a helpful mental health counselling as...,I've been grappling with the aftermath of a tr...,Your experience with a traumatic event is undo...
3,You are a helpful mental health counselling as...,She's been maintaining a stable vital sign rat...,It's essential to acknowledge the progress you...
4,You are a helpful mental health counselling as...,I've been feeling lost and uncertain about my ...,I empathize with your feelings of uncertainty ...


In [6]:
print(df.columns)

Index(['instruction', 'input', 'output'], dtype='object')


In [12]:
def build_prompt(instruction, input_text):

    if pd.isna(input_text) or input_text.strip() == "":
        user_text = instruction
    else:
        user_text = f"{instruction}\n\nUser input: {input_text}"

    return f"""[INST]
You are a helpful, safe, and empathetic mental health assistant.

Respond appropriately and supportively.
Respond ONLY with the assistant's reply.
Do NOT repeat the user input.
Do NOT repeat instructions.

{user_text}
[/INST]
"""

In [13]:
def generate_response(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [14]:
control_outputs = []

for i, row in df.iterrows():

    # THIS is where CSV is consumed
    instruction = row["instruction"]
    input_text = row["input"]

    prompt = build_prompt(instruction, input_text)
    response = generate_response(prompt)

    control_outputs.append(response)

    if (i + 1) % 10 == 0:
        print(f"Completed {i+1}/100")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Completed 10/100


[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Completed 20/100


[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

...

Completed 80/100


[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Completed 90/100


[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Completed 100/100


In [18]:
df["control_output"] = control_outputs

df.to_csv("control_outputs.csv", index=False)

In [19]:
import pandas as pd

df = pd.read_csv("control_outputs.csv")

print(df.shape)
df.head()

(100, 4)


,instruction,input,output,control_output
0,You are a helpful mental health counselling as...,I've been feeling anxious and on edge for week...,I understand that you're feeling anxious and o...,"[INST]\nYou are a helpful, safe, and empatheti..."
1,You are a helpful mental health counselling as...,I've been feeling guilty for not being able to...,I understand that you're feeling guilty for no...,"[INST]\nYou are a helpful, safe, and empatheti..."
2,You are a helpful mental health counselling as...,I've been grappling with the aftermath of a tr...,Your experience with a traumatic event is undo...,"[INST]\nYou are a helpful, safe, and empatheti..."
3,You are a helpful mental health counselling as...,She's been maintaining a stable vital sign rat...,It's essential to acknowledge the progress you...,"[INST]\nYou are a helpful, safe, and empatheti..."
4,You are a helpful mental health counselling as...,I've been feeling lost and uncertain about my ...,I empathize with your feelings of uncertainty ...,"[INST]\nYou are a helpful, safe, and empatheti..."
